# 02 -- Click

## Objectifs pedagogiques

A la fin de ce notebook, vous saurez :

- creer des CLI elegantes avec Click ;
- utiliser les decorateurs `@click.command`, `@click.option`, `@click.argument` ;
- organiser en groupes et sous-commandes ;
- gerer types, callbacks et validation ;
- proposer des prompts et confirmations ;
- colorer la sortie avec `click.style`/`click.echo` ;
- acceder a `click.Context` et passer des donnees partagees ;
- tester avec `CliRunner`.

## Prerequis -- ce que vous devez maitriser

- fonctions typees, decorateurs ;
- la notion de CLI.

Ce que ce notebook **ne couvrira pas** :

- Typer (construit au-dessus de Click) ;
- Rich (pour la mise en forme).

## Plan

1. Installation
2. Premiere commande
3. Arguments et options
4. Types
5. Flags et compteurs
6. Prompts
7. Groupes et sous-commandes
8. Context
9. Callbacks et validation
10. Output colore
11. File handling
12. Tester avec CliRunner
13. Synthese
14. Exercices

---

## 1. Installation

In [ ]:
# pip install click
import click
click.__version__

---

## 2. Premiere commande

In [ ]:
@click.command()
@click.argument('name')
def hello(name: str) -> None:
    click.echo(f'Hello {name}')

# En fichier : if __name__ == '__main__': hello()
print('commande definie')

---

## 3. Arguments et options

In [ ]:
@click.command()
@click.argument('src')
@click.argument('dst')
@click.option('--force', '-f', is_flag=True)
def copy(src: str, dst: str, force: bool) -> None:
    click.echo(f'cp {src} -> {dst}, force={force}')

print('copy definie')

---

## 4. Types

In [ ]:
@click.command()
@click.option('--count', type=int, default=1)
@click.option('--rate', type=click.FloatRange(0.0, 1.0))
@click.option('--out', type=click.Path())
def demo(count: int, rate: float, out: str) -> None:
    click.echo(f'{count=} {rate=} {out=}')

print('types')

### Types Click natifs

| Type | Description |
|---|---|
| `int`, `float`, `str`, `bool` | primitifs |
| `click.IntRange(0, 100)` | entier borne |
| `click.FloatRange(0, 1)` | flottant borne |
| `click.Choice(['a','b'])` | choix restreints |
| `click.Path(exists=True)` | chemin |
| `click.File('r')` | fichier ouvert |
| `click.DateTime(['%Y-%m-%d'])` | datetime |
| `click.UUID` | uuid |

---

## 5. Flags et compteurs

In [ ]:
@click.command()
@click.option('-v', '--verbose', count=True)
@click.option('--debug/--no-debug', default=False)
def run(verbose: int, debug: bool) -> None:
    click.echo(f'verbose={verbose}, debug={debug}')

print('flags')

---

## 6. Prompts

In [ ]:
@click.command()
@click.option('--name', prompt='Votre nom')
@click.option('--password', prompt=True, hide_input=True, confirmation_prompt=True)
def login(name: str, password: str) -> None:
    click.echo(f'Bonjour {name}')

print('prompts definis (appeler en interactif)')

In [ ]:
# Confirmation
# click.confirm('Etes-vous sur ?', abort=True)
print('click.confirm demande oui/non')

---

## 7. Groupes et sous-commandes

In [ ]:
@click.group()
def cli() -> None:
    pass

@cli.command()
@click.argument('name')
def greet(name: str) -> None:
    click.echo(f'Hello {name}')

@cli.command()
def version() -> None:
    click.echo('v1.0')

print('group cli avec 2 commandes')

In [ ]:
# Groupe imbrique
@cli.group()
def db() -> None:
    pass

@db.command()
def migrate() -> None:
    click.echo('migration...')

@db.command()
def seed() -> None:
    click.echo('seed...')

print('cli db migrate / seed')

---

## 8. Context

In [ ]:
@click.group()
@click.option('--verbose', is_flag=True)
@click.pass_context
def app(ctx: click.Context, verbose: bool) -> None:
    ctx.ensure_object(dict)
    ctx.obj['verbose'] = verbose

@app.command()
@click.pass_context
def show(ctx: click.Context) -> None:
    click.echo(f'verbose : {ctx.obj["verbose"]}')

print('Context partage')

---

## 9. Callbacks et validation

In [ ]:
def validate_port(ctx: click.Context, param: click.Parameter, value: int) -> int:
    if not 1024 <= value <= 65535:
        raise click.BadParameter('port doit etre entre 1024 et 65535')
    return value

@click.command()
@click.option('--port', type=int, callback=validate_port, default=8000)
def serve(port: int) -> None:
    click.echo(f'serving on {port}')

print('serve definie')

---

## 10. Output colore

In [ ]:
click.echo(click.style('Succes', fg='green', bold=True))
click.echo(click.style('Erreur', fg='red'))
click.echo(click.style('Info', fg='blue'))
click.echo(click.style('BG', bg='yellow', fg='black'))

In [ ]:
# secho = echo + style
click.secho('Attention !', fg='yellow', bold=True)

---

## 11. File handling

In [ ]:
@click.command()
@click.argument('input_file', type=click.File('r'))
@click.argument('output_file', type=click.File('w'))
def convert(input_file, output_file) -> None:
    for line in input_file:
        output_file.write(line.upper())

print('convert gere - et / comme stdin/stdout')

---

## 12. Tester avec CliRunner

In [ ]:
from click.testing import CliRunner

@click.command()
@click.argument('name')
def hi(name: str) -> None:
    click.echo(f'Hi {name}')

runner = CliRunner()
result = runner.invoke(hi, ['Alice'])
print('exit_code :', result.exit_code)
print('output    :', result.output.strip())

In [ ]:
# Tester un input interactif
@click.command()
def ask() -> None:
    name = click.prompt('Nom')
    click.echo(f'Hello {name}')

result = runner.invoke(ask, input='Bob\n')
print(result.output)

---

## Synthese

| Element | Click |
|---|---|
| Commande | `@click.command()` |
| Positionnel | `@click.argument('name')` |
| Option | `@click.option('--opt', '-o')` |
| Flag | `is_flag=True` ou `--opt/--no-opt` |
| Type | `type=int`, `click.IntRange`, `click.Choice` |
| Prompt | `prompt=True` |
| Groupe | `@click.group()` + `@cli.command()` |
| Context | `@click.pass_context`, `ctx.obj` |
| Couleurs | `click.style`, `click.secho` |
| Fichier | `click.File('r')` |
| Tests | `CliRunner().invoke(cmd, args)` |

### Regles a retenir

- **Decorateurs** : `command`, `group`, `option`, `argument`.
- **Context** pour partager donnees entre commandes.
- **Types** Click valident et convertissent.
- **CliRunner** pour tests sans fork.

---

## Exercices

### Exercice 1 (facile) -- Hello CLI

Creez une commande `hello` qui prend un argument `name` et une option `--greeting` (defaut 'Hello').

In [ ]:
import click

# TODO

<details>
<summary>Voir la correction</summary>

```python
import click

@click.command()
@click.argument('name')
@click.option('--greeting', default='Hello')
def hello(name: str, greeting: str) -> None:
    click.echo(f'{greeting} {name}')

from click.testing import CliRunner
print(CliRunner().invoke(hello, ['Alice', '--greeting', 'Bonjour']).output)
```

</details>

### Exercice 2 (facile) -- Choice restreint

Commande avec option `--format` parmi `json`, `yaml`, `xml`.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
import click

@click.command()
@click.option('--format', 'fmt', type=click.Choice(['json', 'yaml', 'xml']), default='json')
def export(fmt: str) -> None:
    click.echo(f'format : {fmt}')
```

</details>

### Exercice 3 (moyen) -- Groupe database

Creez un groupe `db` avec 3 sous-commandes : `init`, `migrate`, `reset`. Chacune affiche un message.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
import click

@click.group()
def db() -> None:
    pass

@db.command()
def init() -> None:
    click.echo('init db')

@db.command()
def migrate() -> None:
    click.echo('migrating...')

@db.command()
def reset() -> None:
    click.echo('reset !')
```

</details>

### Exercice 4 (moyen) -- Commande avec context partage

Groupe `app` avec option `--verbose`, passee aux sous-commandes via le context.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
import click

@click.group()
@click.option('--verbose', is_flag=True)
@click.pass_context
def app(ctx: click.Context, verbose: bool) -> None:
    ctx.ensure_object(dict)
    ctx.obj['verbose'] = verbose

@app.command()
@click.pass_context
def status(ctx: click.Context) -> None:
    if ctx.obj['verbose']:
        click.echo('status (verbose)')
    else:
        click.echo('status')
```

</details>

### Exercice 5 (difficile) -- Commande convert avec validation

Commande `convert` qui prend un fichier en entree (type `click.Path(exists=True)`), un `--format` parmi `upper`, `lower`, `title`, et une option `--output` pour le fichier de sortie (defaut stdout). Valide avec un callback que l'input est non vide.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
import click
import pathlib

def not_empty(ctx, param, value):
    p = pathlib.Path(value)
    if p.stat().st_size == 0:
        raise click.BadParameter('fichier vide')
    return value

@click.command()
@click.argument('input_file', type=click.Path(exists=True), callback=not_empty)
@click.option('--format', 'fmt', type=click.Choice(['upper', 'lower', 'title']), required=True)
@click.option('--output', type=click.File('w'), default='-')
def convert(input_file: str, fmt: str, output) -> None:
    content = pathlib.Path(input_file).read_text()
    transformed = getattr(content, fmt)()
    output.write(transformed)
```

</details>

---

## Ressources externes

- [Click documentation](https://click.palletsprojects.com/)
- [Click tutorial](https://click.palletsprojects.com/en/latest/quickstart/)
- [Cookiecutter pyclick](https://github.com/pallets/click/tree/main/examples)